In [1]:
'''
abbreviation notes: TDE (tidal disruption event), TC (TDE catalog), PSC (post-starburst catalog),
    SMBHB (supermassive black hole binary)
'''

'\nabbreviation notes: TDE (tidal disruption event), TC (TDE catalog), PSC (post-starburst catalog),\n    SMBHB (supermassive black hole binary)\n'

In [1]:
import os
import otter

import astropy
from astropy.coordinates import SkyCoord
from astropy import units as u

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sparcl.client

_orig_init = sparcl.client.SparclClient.__init__

def _patched_init(self, *args, **kwargs):
    kwargs.setdefault("connect_timeout", 15)
    kwargs.setdefault("read_timeout", 600)
    kwargs.setdefault("announcement", False)   # skips the request that's failing
    _orig_init(self, *args, **kwargs)

sparcl.client.SparclClient.__init__ = _patched_init

In [9]:
db = otter.Otter()

Attempting to login to https://otter.idies.jhu.edu/api with the following credentials:
username: user-guest
password: test


In [8]:
name_list, lum_list, rs_list = [], [], []
cord_ra_list, cord_dec_list, confidence = [], [], []
hosts_boolean, host_names, hosts_cords = [], [], []


# iterate through all transients returned by initial query
i = 0
while i < len(cands):
    object = cands[i]
    
    name = object.default_name
    name_list.append(name)

    
    dists = pd.DataFrame(object['distance']) # find all possible distance measurments
    # luminosity
    lum_dists = dists[dists.distance_type == 'luminosity']
    if len(lum_dists) != 0:
        obj_lums = lum_dists['value'].tolist()
    else:
        obj_lum = 'Empty'
    lum_list.append(obj_lums)
    # redshift
    rs_dists = dists[dists.distance_type == 'redshift']
    obj_rs = (rs_dists['value']).tolist()
    rs_list.append(obj_rs)

    # ra and dec of transient in deg
    cords = object.get_skycoord()
    ra = cords.ra.deg
    cord_ra_list.append(ra)
    dec = cords.dec.deg
    cord_dec_list.append(dec)
    
    # confidence transient is a TDE
    conf = object.get_classification()
    confidence.append(conf[1]) # filter for confidences > 1
    
    info = object.get_host()
    if info == []:
        hosts_boolean.append('False')
        host_names.append('N/A')
        hosts_cords.append('N/A')
    else:
        info = info[0]
        hosts_boolean.append('True')
        host_names.append(info.name)
        hosts_cords.append(info.coord) 
        
    i += 1

    
# initialize data frame storing all possible canidates
cands_df = pd.DataFrame(
    {"TDE Name": name_list, "default redshift": [item[0] for item in rs_list], "redshifts": rs_list, "luminosity (Mpc)": lum_list, 
     "confidence": confidence, "ra (deg)": cord_ra_list, "dec (deg)":cord_dec_list, "Host info in OTTER?": hosts_boolean, 
     "Host Name": host_names, "Host Coords (deg)": hosts_cords}
)

cands_df = cands_df.set_index("TDE Name")

cands_df

,default redshift,redshifts,luminosity (Mpc),confidence,ra (deg),dec (deg),Host info in OTTER?,Host Name,Host Coords (deg)
TDE Name,,,,,,,,,
NGC 5905,0.01124,"[0.01124, 0.012]",[50.17],3.1,228.847167,55.517108,True,NGC 5905,"<SkyCoord (ICRS): (ra, dec) in deg\n (228.8..."
ARP 299-B AT1,0.010411,[0.010411],[50.17],3.3,172.129115,58.561329,False,N/A,N/A
XMMSL1 J0740-85,0.0173,[0.0173],[77.6],3.3,115.034125,-85.658681,True,2MASX 07400785-8539307,"<SkyCoord (ICRS): (ra, dec) in deg\n (115.0..."
XMMSL2 J144605.0+685735,0.029,[0.029],[77.6],3.3,221.520833,68.958611,True,"2MASX 14460522+6857311, LEDA 2725953","<SkyCoord (ICRS): (ra, dec) in deg\n (221.5..."
2016fnl,0.0163,"[0.0163, 0.016328]",[73],3.3,7.487667,32.893750,True,MRK 950,"<SkyCoord (ICRS): (ra, dec) in deg\n (7.487..."
2022gri,0.028,[0.028],[73],3.3,109.586643,33.994853,False,N/A,N/A
2022dsb,0.023,"[0.023, 0.0234]",[73],3.3,235.590633,-22.670580,False,N/A,N/A
2019azh,0.02199999988079071,"[0.02199999988079071, 0.022, 0.0222, 0.022346]",[73],3.3,123.320627,22.648301,True,KUG 0810+227,"<SkyCoord (ICRS): (ra, dec) in deg\n (123.3..."
2020wey,0.02738,"[0.02738, 0.02741]",[73],3.3,136.357958,61.802538,False,N/A,N/A


Attempting to login to https://otter.idies.jhu.edu/api with the following credentials:
username: user-guest
password: test


In [3]:
# note: only concerned with TC for now

# initialize canidate list as objects classified as TDEs and within redshift 0.03
cands = db.query(classification="TDE", minz=0, maxz=0.03, class_confidence_threshold=1)

# initalize lists to be stored in dataframe
name_list = []

rs_list = []
lum_list = []

cord_ra_list = []
cord_dec_list = []

confidence = []

host_names = []
hosts_cords = []

SPARCL_flux = []
SPARCL_wl = []
SPARCL_specid = []
SPARCL_dr = []

In [4]:
'''
helper functions
'''

def no_hosts():
    host_names.append('Unknown!')
    hosts_cords.append('N/A')
    SPARCL_flux.append('N/A')
    SPARCL_wl.append('N/A')
    SPARCL_specid.append('N/A')
    SPARCL_dr.append('N/A')


def get_host_helper(transient):
    info = transient.get_host()
    if info == []:
        return False
    else:
        host_names.append(info[0].name)
        hosts_cords.append((info[0].coord.ra.deg, info[0].coord.dec.deg))
        return info

In [7]:
'''
main function
returns list of potential galaxy canidates
'''

# iterate through all transients returned by initial query
i = 0
while i < len(cands):
    object = cands[i]
    
    name = object.default_name
    name_list.append(name)

    
    dists = pd.DataFrame(object['distance']) # find all possible distance measurments
    # luminosity
    lum_dists = dists[dists.distance_type == 'luminosity']
    if len(lum_dists) != 0:
        obj_lums = lum_dists['value'].tolist()
    else:
        obj_lum = 'Empty'
    lum_list.append(obj_lums)
    # redshift
    rs_dists = dists[dists.distance_type == 'redshift']
    obj_rs = (rs_dists['value']).tolist()
    rs_list.append(obj_rs)

    # ra and dec of transient in deg
    cords = object.get_skycoord()
    ra = cords.ra.deg
    cord_ra_list.append(ra)
    dec = cords.dec.deg
    cord_dec_list.append(dec)
    
    # confidence transient is a TDE
    conf = object.get_classification()
    confidence.append(conf[1]) # filter for confidences > 1
    
    info = get_host_helper(object)
    if info == False:
        no_hosts()
    else:
        try:
            query = info[0].query_sparcl()
            for row in query:
                SPARCL_flux.append(row['flux'])
                SPARCL_wl.append(row['wavelength'])
                SPARCL_specid.append(row['specid'])
                SPARCL_dr.append(row['_dr'])
        except:
            SPARCL_flux.append('N/A')
            SPARCL_wl.append('N/A')
            SPARCL_specid.append('Host not in SPARCL')
            SPARCL_dr.append('N/A')

    i += 1


# initialize data frame storing all possible canidates
cands_df = pd.DataFrame(
    {"TDE Name": name_list, "default redshift": [item[0] for item in rs_list], "redshifts": rs_list, "luminosity (Mpc)": lum_list, 
     "confidence": confidence, "ra (deg)": cord_ra_list, "dec (deg)":cord_dec_list, "Host Name": host_names, 
     "Host Coords (deg)": hosts_cords, "Spec ID (SPARCL)": SPARCL_specid, "DR (SPARCL)": SPARCL_dr, "Flux (SPARCL)": SPARCL_flux, 
     "Wavelength (SPARCL)": SPARCL_wl}
)

cands_df = cands_df.set_index("TDE Name")

cands_df

Object not found in Sparcl!


ValueError: All arrays must be of the same length

In [20]:
'''
main function
returns list of potential galaxy canidates
'''
# note: only concerned with TC for now

# initialize canidate list as objects classified as TDEs and within redshift 0.03
cands = db.query(classification="TDE", minz=0, maxz=0.03, class_confidence_threshold=1)

# initalize lists to be stored in dataframe
name_list = []

rs_list = []
lum_list = []

cord_ra_list = []
cord_dec_list = []

confidence = []

host_names = []
hosts_cords = []

SPARCL_flux = []
SPARCL_wl = []
SPARCL_specid = []
SPARCL_dr = []

# iterate through all transients returned by initial query
i = 0
while i < len(cands):
    object = cands[i]
    
    name = object.default_name
    name_list.append(name)

    
    dists = pd.DataFrame(object['distance']) # find all possible distance measurments
    # luminosity
    lum_dists = dists[dists.distance_type == 'luminosity']
    if len(lum_dists) != 0:
        obj_lums = lum_dists['value'].tolist()
    else:
        obj_lum = 'Empty'
    lum_list.append(obj_lums)
    # redshift
    rs_dists = dists[dists.distance_type == 'redshift']
    obj_rs = (rs_dists['value']).tolist()
    rs_list.append(obj_rs)

    # ra and dec of transient in deg
    cords = object.get_skycoord()
    ra = cords.ra.deg
    cord_ra_list.append(ra)
    dec = cords.dec.deg
    cord_dec_list.append(dec)
    
    # confidence transient is a TDE
    conf = object.get_classification()
    confidence.append(conf[1]) # filter for confidences > 1
    
    info = object.get_host()[0] # does OTTER have an object of class host assosiated with the transient?
    # need to know this to know if OTTER can return photometric information
    if info == None:
        host_names.append('Unknown!')
        hosts_cords.append('N/A')
        SPARCL_flux.append('N/A')
        SPARCL_wl.append('N/A')
        SPARCL_specid.append('N/A')
        SPARCL_dr.append('N/A')
    else:
        host_names.append(info.name)
        hosts_cords.append((info.coord.ra.deg, info.coord.dec.deg))

        spec = info.query_sparcl() # spectroscopy data from SPARCL
        for row in spec:
            SPARCL_flux.append(row['flux'])
            SPARCL_wl.append(row['wavelength'])
            SPARCL_specid.append(row['specid'])
            SPARCL_dr.append(row['_dr'])

    i += 1


# initialize data frame storing all possible canidates
cands_df = pd.DataFrame(
    {"TDE Name": name_list, "default redshift": [item[0] for item in rs_list], "redshifts": rs_list, "luminosity (Mpc)": lum_list, 
     "confidence": confidence, "ra (deg)": cord_ra_list, "dec (deg)":cord_dec_list, "Host Name": host_names, 
     "Host Coords (deg)": hosts_cords, "DR (SPARCL)": SPARCL_dr, "Flux (SPARCL)": SPARCL_flux, 
     "Wavelength (SPARCL)": SPARCL_wl, "Spec ID (SPARCL)": SPARCL_specid}
)

cands_df = cands_df.set_index("TDE Name")

cands_df

Object not found in Sparcl!


TypeError: 'NoneType' object is not iterable

In [14]:
first_row = cands_df.iloc[0]

first_row

redshift                  0.01124
confidence                    3.1
ra                     228.847167
dec                     55.517108
Host info in OTTER?          True
Name: NGC 5905, dtype: object